# Double Factorization

Bloqs for the double-factorized chemistry Hamiltonian in second quantization.

Recall that for the single factorized Hamiltonian we have
$$
    H = \sum_{pq}T'_{pq} a_p^\dagger a_q + \frac{1}{2} \sum_l \left(\sum_{pq}
W_{pq}^{(l)} a_p^\dagger a_q\right)^2.
$$
One arrives at the double factorized (DF) Hamiltonian by further factorizing the
$W_{pq}^{(l)}$ terms as
$$
    W^{(l)}_{pq} = \sum_{k} U^{(l)}_{pk} f_k^{(l)} U^{(l)*}_{qk},
$$
so that
$$
    H = \sum_{pq}T'_{pq} a_p^\dagger a_q + \frac{1}{2} \sum_l U^{(l)}\left(\sum_{k}^{\Xi^{(l)}}
        f_k^{(l)} n_k\right)^2 U^{(l)\dagger}
$$
where $\Xi^{(l)} $ is the rank of second factorization.

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

The two original examples declare ports and resources at 50 auxiliaries,
10 spin orbitals and 250 eigenvalues. The final cell binds a smaller
coefficient fixture through the executable interface.

In [ ]:
from dataclasses import asdict
from io import BytesIO

import matplotlib.pyplot as plt
from IPython.display import Image, SVG, display

from examples.algorithms.double_factorization_resources import (
    notebook_block_score, notebook_models, notebook_signatures,
)
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

## Double-factorized one-body block

The original example models inner preparation, spin selection, programmed
basis rotations and the number phase. The return stages undo the basis
rotations, spin exchange and preparation. The number phase is controlled
by the two preparation-success flags in the locked decomposition.
The separate factor selector is present as a through port.

In [ ]:
from flagquantum.algorithms.primitives.double_factorization_encoding_model import (
    source_double_factorization_one_body_cost,
)

### Example Instances

In [ ]:
one_body, block = notebook_models()
one_body_signature, block_signature = notebook_signatures()
assert one_body == source_double_factorization_one_body_cost(
    10, 250, alias_bits=12, basis_rotation_bits=7,
)
assert one_body.qubits == one_body_signature.n_qubits()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("DF one-body", one_body_signature, {})])))

### Call Graph

In [ ]:
def show_resource_graph(model):
    dot = export_modeled_call_graph_dot(model, max_depth=1)
    display(SVG(render_call_graph_svg(dot)))
    report = estimate_ftqc_resources(model)
    print("Logical gate estimate", asdict(report.logical.gates))
    print("Peak logical qubits", report.logical.peak_logical_qubits)
    assert report.emitted is None


show_resource_graph(one_body)

## Full double-factorized block

Prepare an auxiliary index, load its rank and rotation data, execute the
one-body block, reflect the inner address and spin, then repeat the
one-body block and release the prepared data. The source reflection
uses the outer success flag. The register score shows its actual three
port groups in the frozen decomposition. The logical estimates use the
original placeholder lookup costs and their direct call ordering.

In [ ]:
from flagquantum.algorithms.primitives.double_factorization_encoding_model import (
    source_double_factorization_block_encoding_cost,
)

### Example Instances

In [ ]:
assert block == source_double_factorization_block_encoding_cost(
    50, 10, 250, alias_bits=12, basis_rotation_bits=7, inner_rotation_bits=7,
)
assert block.qubits == block_signature.n_qubits()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("DF block encoding", block_signature, {})])))

In [ ]:
score = notebook_block_score()
figure, _ = draw_register_score(register_score_data(score), unit_to_inches=1.4)
buffer = BytesIO()
figure.savefig(buffer, format="png", bbox_inches="tight")
plt.close(figure)
display(Image(data=buffer.getvalue()))

### Call Graph

In [ ]:
show_resource_graph(block)

### Append a small concrete block

Bind two spin orbitals and one factor row using the usual append path.
The table-backed executable has explicit workspace. Read the physical
instruction counts after append.

In [ ]:
from flagquantum.algorithms.primitives.double_factorization_block_encoding import (
    DoubleFactorizationBlockEncoding,
)
from flagquantum.compiler._component_trace.resource_comparison import count_emitted_instructions
from flagquantum.compiler.registers.layout.partition import partition_register
from flagquantum.dynamic import DynamicCircuit

encoding = DoubleFactorizationBlockEncoding.from_eigenvalues(
    (1.0,), ((2.0,),), ((), ()), outer_bits=1, inner_bits=1,
    rank_rotation_bits=2, basis_rotation_bits=4,
)
logical = sum(width for _, width in encoding.layout)
work, slots = encoding.requirements()
ports = partition_register(tuple(range(logical)), encoding.layout)
circuit = DynamicCircuit(logical + work)
circuit.append(encoding.make_on(ports, workspace=tuple(range(logical, logical + work))))
physical = count_emitted_instructions(circuit.to_ir())
assert physical.unconditional["measure"] >= slots
print("Emitted T/T-dagger", physical.t_count)
print("Emitted measurements", physical.unconditional["measure"])
print("Reserved wires", circuit.n_qubits)